# B — BVNR Unit-Aware Data Pipeline

This notebook demonstrates how to parse `.bvnr` files with physical-unit
annotations, flatten numeric measurements into a pandas DataFrame, filter
by SI dimension, and produce unit-aware matplotlib plots.

**Requires:** `libbvnr_shared.so` on `LIBBOVNAR_PATH`.

## Contents
1. [Build a sample sensor document](#1.-Build-a-sample-sensor-document)
2. [Flatten to DataFrame](#2.-Flatten-to-DataFrame)
3. [Filter by SI dimension](#3.-Filter-by-SI-dimension)
4. [Unit-aware bar chart](#4.-Unit-aware-bar-chart)
5. [Unit conversion](#5.-Unit-conversion)
6. [Dimension labels](#6.-Dimension-labels)


## Setup


In [ ]:
import os
import matplotlib.pyplot as plt

import bovnar
from bovnar.analytics import (
    doc_to_dataframe,
    filter_by_dim_name,
    plot_scalar_series,
    dim_label,
    SI_DIM_NAMES,
)
from bovnar.units import (
    unit_dimension_vector,
    units_compatible,
    convert_value,
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 110


## 1. Build a sample sensor document

A realistic sensor-node configuration with physical units on every
numeric field.  In production this would be read from a `.bvnr` file
written by the embedded device.


In [ ]:
from bovnar.writer import Writer
from bovnar.enums import BaseUnit, SIPrefix

with Writer.to_mem() as w:
    w.write_uint("node_id",    7,     width=8)
    w.write_float("frequency", 868.1, width=64,
                  unit_si_base=BaseUnit.HERTZ,
                  unit_si_prefix=SIPrefix.MEGA)
    w.write_sint("tx_power",   14,    width=8,
                 unit_si_base=BaseUnit.WATT)   # dBm stored as W here for demo
    w.write_uint("interval",   60,    width=32,
                 unit_si_base=BaseUnit.SECOND)

    # Nested struct: calibration
    w.begin_struct("calibration")
    w.write_float("offset", -0.5,  width=64,
                  unit_si_base=BaseUnit.KELVIN)  # offset in Kelvin
    w.write_float("gain",    1.002, width=64)
    w.end_struct()

bvnr_bytes = w.get_output()
print(bvnr_bytes.decode())


Parse and inspect the DOM:


In [ ]:
doc = bovnar.dom_parse(bvnr_bytes)
print(f'Top-level keys: {[k for k, _ in doc]}')

freq_node = doc['frequency']
print(f'frequency unit_str  : {freq_node.unit_str}')
print(f'frequency value_si  : {freq_node.value_in_base_units():.0f} Hz')


## 2. Flatten to DataFrame

`doc_to_dataframe` traverses the document recursively.  Struct nesting
is reflected as a dot-separated `path`.  The `dims` column holds the
7-element SI dimension exponent vector `[m, kg, s, A, K, mol, cd]`.


In [ ]:
df = doc_to_dataframe(doc)
df


In [ ]:
# Human-readable dimension label for each row
df['dimension'] = df['dims'].apply(dim_label)
df[["path", "value_si", "unit_str", "dimension"]]


## 3. Filter by SI dimension

`filter_by_dim_name` selects rows whose SI dimension vector matches a
named physical dimension.  This works across any unit prefix: a field
annotated `<float:64,M-Hz>` and one annotated `<float:64,k-Hz>` both
match `"frequency"` because they share the same dimension vector
`[0, 0, -1, 0, 0, 0, 0]`.


In [ ]:
freq_rows = filter_by_dim_name(df, 'frequency')
print("Frequency measurements:")
print(freq_rows[["path", "value_si", "unit_str"]])

time_rows = filter_by_dim_name(df, 'time')
print("\nTime measurements:")
print(time_rows[["path", "value_si", "unit_str"]])


## 4. Unit-aware bar chart

`plot_scalar_series` groups measurements by `unit_str` and places each
group on a separate subplot so incompatible units never share an axis.


In [ ]:
fig = plot_scalar_series(
    df,
    x_col="path",
    y_col="value_si",
    group_col="unit_str",
    title="Sensor node measurements (SI base units)",
)
plt.show()


## 5. Unit conversion

`convert_value` handles both multiplicative and affine conversions
(e.g. Celsius → Kelvin → Fahrenheit) without manual formula lookup.


In [ ]:
# Build a small multi-temperature document
temp_bvnr = (
    b".ambient   = <float:64,K>   295.15;\n"  # Kelvin
    b".setpoint  = <float:64,K>   373.15;\n"  # 100 °C in K
)
tdoc = bovnar.dom_parse(temp_bvnr)

ambient_unit  = tdoc["ambient"].unit
setpoint_unit = tdoc["setpoint"].unit

# Parse a Celsius unit for conversion target
celsius_unit = bovnar.parse_unit("°C")

print("Compatibility check (both are temperature):",
      units_compatible(ambient_unit, celsius_unit))

ambient_k  = tdoc["ambient"].value_in_base_units()   # already K
ambient_c  = convert_value(ambient_k,  ambient_unit, celsius_unit)
setpoint_c = convert_value(
    tdoc["setpoint"].value_in_base_units(), setpoint_unit, celsius_unit
)

print(f"Ambient  : {ambient_k:.2f} K  →  {ambient_c:.2f} °C")
print(f"Setpoint : {tdoc['setpoint'].value_in_base_units():.2f} K  →  {setpoint_c:.2f} °C")


## 6. Dimension labels

The `dim_label` function covers named SI dimensions.  For exotic
compound units not in its lookup table it falls back to an algebraic
Unicode expression.


In [ ]:
examples = [
    [1, 0, -1, 0, 0, 0, 0],   # velocity
    [1, 1, -2, 0, 0, 0, 0],   # force
    [2, 1, -2, 0, 0, 0, 0],   # energy
    [0, 0, -1, 0, 0, 0, 0],   # frequency
    [-1, 1, -2, 0, 0, 0, 0],  # pressure
    [3, 0, -3, -1, 0, 0, 0],  # unknown compound — algebraic fallback
]

header = "{:<35} {}".format("Dimension vector", "Label")
print(header)
print("-" * 55)
for d in examples:
    print("{:<35} {}".format(str(d), dim_label(d)))
